In [ ]:
!pip install segmentation-models-pytorch albumentations earthengine-api folium geemap -q

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

import folium
from folium.plugins import TimestampedGeoJson
import json
import warnings
warnings.filterwarnings('ignore')


In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Layer, Conv2D, BatchNormalization, ReLU

MODEL_PATH = Path('/content/drive/MyDrive/Codes/model/deeplabv3_50.keras')

In [ ]:
# Custom objects: must match the training notebook exactly
reg = tf.keras.utils.register_keras_serializable(package='glof')

@reg
class ResNetPreprocess(Layer):
    def call(self, x):
        x = tf.reverse(x, axis=[-1])
        return x - tf.constant([103.939, 116.779, 123.68], dtype=x.dtype)

@reg
class ConvBlock(Layer):
    def __init__(self, filters=256, kernel_size=3, dilation_rate=1, **kwargs):
        super().__init__(**kwargs)
        self.filters, self.kernel_size, self.dilation_rate = filters, kernel_size, dilation_rate
        self.net = Sequential([
            Conv2D(filters, kernel_size=kernel_size, padding='same', dilation_rate=dilation_rate,
                   use_bias=False, kernel_initializer='he_normal'),
            BatchNormalization(),
            ReLU(),
        ])
    def call(self, X):
        return self.net(X)
    def get_config(self):
        return {**super().get_config(), 'filters': self.filters,
                'kernel_size': self.kernel_size, 'dilation_rate': self.dilation_rate}

@reg
def dice_loss(y_true, y_pred, smooth=1.0):
    y_true = tf.cast(y_true, y_pred.dtype)
    inter = tf.reduce_sum(y_true * y_pred, axis=[1, 2, 3])
    denom = tf.reduce_sum(y_true, axis=[1, 2, 3]) + tf.reduce_sum(y_pred, axis=[1, 2, 3])
    return 1.0 - tf.reduce_mean((2.0 * inter + smooth) / (denom + smooth))

@reg
def bce_dice_loss(y_true, y_pred):
    bce = tf.reduce_mean(tf.keras.losses.binary_crossentropy(y_true, y_pred))
    return bce + dice_loss(y_true, y_pred)

@reg
def dice_coefficient(y_true, y_pred, smooth=1.0):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred > 0.5, tf.float32)
    inter = tf.reduce_sum(y_true * y_pred)
    return (2.0 * inter + smooth) / (tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth)

In [ ]:
model = tf.keras.models.load_model(MODEL_PATH, compile=False)
model.summary()

In [ ]:
import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import requests
import io
import zipfile
from pathlib import Path
from PIL import Image

import rasterio
from rasterio.io import MemoryFile

In [ ]:
try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize(project='glof-510607')

print("Google Earth Engine initialized successfully!")

In [ ]:
LAKES = {

    'UK_GL_01': {
        'center': [30.910539, 78.958097],
        'bbox':   [78.933097, 30.890539, 78.983097, 30.930539],
        'name':   'Kedartal, Uttarakhand'
    },

    'UK_GL_02': {
        'center': [30.900911, 79.754922],
        'bbox':   [79.729922, 30.880911, 79.779922, 30.920911],
        'name':   'Vasudhara Tal, Uttarakhand'
    },

    'UK_GL_03': {
        'center': [30.446003, 80.387528],
        'bbox':   [80.362528, 30.426003, 80.412528, 30.466003],
        'name':   'Mabang Tal (Surya Sarovar), Uttarakhand'
    },

    'UK_GL_04': {
        'center': [30.408394, 80.511719],
        'bbox':   [80.486719, 30.388394, 80.536719, 30.428394],
        'name':   'Pyungru Tal, Uttarakhand'
    },

    'UK_GL_05': {
        'center': [30.742489, 78.987669],
        'bbox':   [78.962669, 30.722489, 79.012669, 30.762489],
        'name':   'Masar Tal (Bhilangna Lake), Uttarakhand'
    },

    'UK_GL_06': {
        'center': [30.564339, 80.177542],
        'bbox':   [80.152542, 30.544339, 80.202542, 30.584339],
        'name':   'Safed Tal (Goriganga Lake), Uttarakhand'
    },

    'UK_GL_07': {
        'center': [30.392822, 80.532919],
        'bbox':   [80.507919, 30.372822, 80.557919, 30.412822],
        'name':   'Syantyankti Lake, Uttarakhand'
    },

    'UK_GL_08': {
        'center': [31.151186, 79.267256],
        'bbox':   [79.242256, 31.131186, 79.292256, 31.171186],
        'name':   'Unnamed Lake 1, Uttarakhand'
    },

    'UK_GL_09': {
        'center': [30.455625, 80.515914],
        'bbox':   [80.490914, 30.435625, 80.540914, 30.475625],
        'name':   'Unnamed Lake 2, Uttarakhand'
    },

    'UK_GL_10': {
        'center': [30.263953, 80.712808],
        'bbox':   [80.687808, 30.243953, 80.737808, 30.283953],
        'name':   'Unnamed Lake 3 (Kuthi Yangti Valley Lower), Uttarakhand'
    },

    'UK_GL_11': {
        'center': [30.372439, 80.603647],
        'bbox':   [80.578647, 30.352439, 80.628647, 30.392439],
        'name':   'Unnamed Lake 4 (Kuthi Yangti Valley Upper), Uttarakhand'
    },

    'UK_GL_12': {
        'center': [30.183364, 79.875664],
        'bbox':   [79.850664, 30.163364, 79.900664, 30.203364],
        'name':   'NaagKund (Bhairav Kund), Uttarakhand'
    },

    'UK_GL_13': {
        'center': [30.813242, 79.925497],
        'bbox':   [79.900497, 30.793242, 79.950497, 30.833242],
        'name':   'Saraswati Tal, Uttarakhand'
    }
}


# ---------------------------------------------------------
# Monitoring years
# ---------------------------------------------------------

YEARS = [2021, 2022, 2023, 2024, 2025]

# Sentinel-2 bands used by the trained U-Net
BANDS = ['B8', 'B4', 'B3']


print(f"{len(LAKES)} Uttarakhand lakes selected.")
print(f"Monitoring years: {YEARS}")
print(f"Model bands: {BANDS}")


In [ ]:
# ---------------------------------------------------------
# Sentinel-2 image preparation
# ---------------------------------------------------------

def mask_s2_clouds_shadows(image):
    """
    Mask clouds and cloud shadows using Sentinel-2 Scene
    Classification Layer (SCL).

    SCL classes masked:
        1 = saturated/defective
        3 = cloud shadow
        8 = cloud medium probability
        9 = cloud high probability
        10 = cirrus
    """

    scl = image.select('SCL')

    clear_mask = (
        scl.neq(1)
        .And(scl.neq(3))
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
    )

    return image.updateMask(clear_mask)


def get_sentinel2_image(bbox, year, bands=BANDS):
    """
    Prepare a Sentinel-2 summer composite for one lake and year.

    Returns:
        composite image
        region geometry
        number of source scenes
    """

    region = ee.Geometry.Rectangle(
        bbox,
        proj='EPSG:4326',
        geodesic=False
    )

    start_date = f'{year}-06-01'
    end_date   = f'{year + 1}-03-01'
    # start_date = f'{year}-06-01'
    # end_date   = f'{year}-10-01'

    # First use relatively strict cloud filtering
    collection = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(region)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    )

    count = collection.size().getInfo()

    # Fallback if very few images are available
    if count == 0:
        print(f"    No scenes <20% cloud. Trying <40% cloud...")

        collection = (
            ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(region)
            .filterDate(start_date, end_date)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 40))
        )

        count = collection.size().getInfo()

    if count == 0:
        raise RuntimeError(
            f"No Sentinel-2 images found for {year} "
            f"within bbox {bbox}"
        )

    # Cloud/shadow masking
    collection = collection.map(mask_s2_clouds_shadows)

    # Select only the bands used by the model
    composite = collection.select(bands).median()

    print(f"    Sentinel-2 scenes used: {count}")

    return composite, region

In [ ]:
def get_utm_epsg(lon, lat):
    zone = int((lon + 180) / 6) + 1

    if lat >= 0:
        return 32600 + zone
    else:
        return 32700 + zone

# ---------------------------------------------------------
# Download native Sentinel-2 reflectance
# ---------------------------------------------------------

def download_sentinel2_geotiff(image, region, center_lon, center_lat):

    epsg = get_utm_epsg(center_lon, center_lat)

    params = {
        'region': region,
        'scale': 10,
        'crs': f'EPSG:{epsg}',
        'bands': BANDS,
        'format': 'GEO_TIFF',
        'filePerBand': False
    }

    url = image.getDownloadURL(params)

    response = requests.get(url, timeout=120)
    response.raise_for_status()

    content = response.content

    if content[:2] == b'PK':

        with zipfile.ZipFile(io.BytesIO(content)) as z:

            tif_files = [
                name for name in z.namelist()
                if name.lower().endswith(('.tif', '.tiff'))
            ]

            if not tif_files:
                raise RuntimeError(
                    "No GeoTIFF found in Earth Engine ZIP."
                )

            tif_bytes = z.read(tif_files[0])

    else:
        tif_bytes = content

    return tif_bytes

In [ ]:
# ---------------------------------------------------------
# Read downloaded GeoTIFF
# ---------------------------------------------------------

def geotiff_to_array(tif_bytes):
    """
    Convert GeoTIFF bytes to numpy array.

    Returns:
        img_8bit      : H x W x 3 uint8 image
        transform     : raster affine transform
        crs           : raster CRS
        shape         : original raster dimensions
    """

    with MemoryFile(tif_bytes) as memfile:

        with memfile.open() as src:

            data = src.read()     # bands x height x width

            transform = src.transform
            crs = src.crs

            if data.shape[0] != 3:
                raise ValueError(
                    f"Expected 3 bands, got {data.shape[0]}"
                )

            # Earth Engine Sentinel-2 SR values are scaled.
            # Convert reflectance to approximately 0-1.
            data = data.astype(np.float32) / 10000.0

            # Remove invalid values
            data = np.nan_to_num(
                data,
                nan=0.0,
                posinf=1.0,
                neginf=0.0
            )

            data = np.clip(data, 0, 1)

            # B8/B4/B3 → 3-channel image
            img_float = np.transpose(data, (1, 2, 0))

            # Convert to the 8-bit representation expected by
            # the existing Albumentations/ImageNet pipeline.
            img_8bit = (
                img_float * 255
            ).round().astype(np.uint8)

            return (
                img_8bit,
                transform,
                crs,
                src.height,
                src.width
            )


In [ ]:
val_transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

In [ ]:
# ---------------------------------------------------------
# TensorFlow / Keras inference
# ---------------------------------------------------------

def predict_mask(
    model,
    img_array,
    img_size=400,
    threshold=0.5
):
    """
    Run the trained DeepLabV3+ Keras model.

    Returns:
        probability map
        binary prediction mask
    """

    original_height, original_width = img_array.shape[:2]

    # Resize Sentinel image to training size
    img_resized = cv2.resize(
        img_array,
        (img_size, img_size),
        interpolation=cv2.INTER_LINEAR
    )

    # Keras expects NHWC:
    # (batch, height, width, channels)
    img_batch = np.expand_dims(
        img_resized.astype(np.float32),
        axis=0
    )

    # IMPORTANT:
    # Do NOT use model.eval()
    # Do NOT use torch.no_grad()
    prediction = model.predict(
        img_batch,
        verbose=0
    )

    probability = np.asarray(prediction)

    # Expected shape = (1, 400, 400, 1)
    if probability.ndim == 4:
        probability = probability[0, ..., 0]

    elif probability.ndim == 3:
        probability = probability[0, ...]

    elif probability.ndim != 2:
        raise ValueError(
            f"Unexpected model output shape: {probability.shape}"
        )

    probability = probability.astype(np.float32)

    # Your saved model already has an OutputMask activation.
    # This guard handles logits too, without harming normal probabilities.
    if probability.min() < 0.0 or probability.max() > 1.0:
        probability = 1.0 / (1.0 + np.exp(-probability))

    # Resize probability map back to native Sentinel dimensions
    probability = cv2.resize(
        probability,
        (original_width, original_height),
        interpolation=cv2.INTER_LINEAR
    )

    # Convert probability to binary mask
    mask = (
        probability >= threshold
    ).astype(np.uint8)

    return probability, mask

In [ ]:
# ---------------------------------------------------------
# Area calculation from raster pixel dimensions
# ---------------------------------------------------------
def mask_to_area_km2(mask, transform, crs):

    print(f"    CRS: {crs}")
    print(
        f"    Pixel size: "
        f"{abs(transform.a):.6f} × "
        f"{abs(transform.e):.6f}"
    )

    if crs is None:
        raise ValueError("Raster CRS is missing.")

    # We expect a projected CRS with metre units
    if crs.is_geographic:
        raise ValueError(
            f"Raster CRS {crs} is geographic (degrees). "
            "Area calculation requires a projected CRS in metres."
        )

    pixel_width_m = abs(transform.a)
    pixel_height_m = abs(transform.e)

    pixel_area_m2 = (
        pixel_width_m *
        pixel_height_m
    )

    water_pixels = np.count_nonzero(mask)

    return (
        water_pixels *
        pixel_area_m2 /
        1_000_000
    )

In [ ]:
# ---------------------------------------------------------
# Diagnostic visualization
# ---------------------------------------------------------

def show_prediction(
    img_array,
    probability,
    mask,
    lake_name,
    year
):

    fig, axes = plt.subplots(
        1, 3,
        figsize=(15, 5)
    )

    # Input
    axes[0].imshow(img_array)
    axes[0].set_title(
        f'{lake_name} - {year}\nInput B8/B4/B3'
    )
    axes[0].axis('off')

    # Probability
    im = axes[1].imshow(
        probability,
        vmin=0,
        vmax=1,
        cmap='viridis'
    )
    axes[1].set_title(
        'U-Net Lake Probability'
    )
    axes[1].axis('off')

    plt.colorbar(
        im,
        ax=axes[1],
        fraction=0.046
    )

    # Binary mask
    axes[2].imshow(
        mask,
        cmap='gray'
    )
    axes[2].set_title(
        'Binary Lake Mask'
    )
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


In [ ]:
# ---------------------------------------------------------
# Multi-temporal processing
# Uttarakhand: 13 lakes × 5 years
# ---------------------------------------------------------

results = {
    lake_id: {}
    for lake_id in LAKES
}


total_tasks = len(LAKES) * len(YEARS)
completed_tasks = 0


for lake_id, lake_info in LAKES.items():

    print("\n" + "=" * 80)
    print(f"Processing: {lake_info['name']}")
    print(f"Lake ID   : {lake_id}")
    print(f"Center    : {lake_info['center']}")
    print(f"BBox      : {lake_info['bbox']}")
    print("=" * 80)

    for year in YEARS:

        completed_tasks += 1

        print("\n" + "-" * 70)
        print(
            f"Task {completed_tasks}/{total_tasks} | "
            f"{lake_info['name']} | Year {year}"
        )
        print("-" * 70)

        try:

            # -------------------------------------------------
            # 1. Get Sentinel-2 summer composite
            # -------------------------------------------------

            image, region = get_sentinel2_image(
                lake_info['bbox'],
                year
            )


            # -------------------------------------------------
            # 2. Download native GeoTIFF
            # -------------------------------------------------

            tif_bytes = download_sentinel2_geotiff(
                image,
                region,
                lake_info['center'][1],   # longitude
                lake_info['center'][0]    # latitude
            )


            # -------------------------------------------------
            # 3. Convert GeoTIFF to image array
            # -------------------------------------------------

            (
                img_array,
                transform,
                crs,
                height,
                width
            ) = geotiff_to_array(tif_bytes)


            # -------------------------------------------------
            # 4. U-Net inference
            # -------------------------------------------------

            probability, mask = predict_mask(model,img_array,img_size=400,threshold=0.5)

            # -------------------------------------------------
            # 5. Calculate lake area
            # -------------------------------------------------

            area_km2 = mask_to_area_km2(
                mask,
                transform,
                crs
            )


            # -------------------------------------------------
            # 6. Diagnostic statistics
            # -------------------------------------------------

            lake_fraction = mask.mean() * 100
            water_pixels = np.count_nonzero(mask)

            print(f"Raster size      : {width} × {height}")
            print(f"CRS              : {crs}")
            print(
                f"Pixel size       : "
                f"{abs(transform.a):.4f} × "
                f"{abs(transform.e):.4f} m"
            )
            print(
                f"Probability min  : "
                f"{probability.min():.4f}"
            )
            print(
                f"Probability max  : "
                f"{probability.max():.4f}"
            )
            print(
                f"Probability mean : "
                f"{probability.mean():.4f}"
            )
            print(
                f"Lake pixels      : "
                f"{water_pixels:,}"
            )
            print(
                f"Lake fraction    : "
                f"{lake_fraction:.2f}%"
            )
            print(
                f"Lake area        : "
                f"{area_km2:.4f} km²"
            )


            # -------------------------------------------------
            # 7. Store results
            # -------------------------------------------------

            results[lake_id][year] = {

                'image': img_array,

                'probability': probability,

                'mask': mask,

                'area_km2': area_km2,

                'transform': transform,

                'crs': str(crs),

                'height': height,

                'width': width,

                'water_pixels': water_pixels,

                'lake_fraction': lake_fraction,

                'error': None

            }


        except Exception as e:

            print(
                f"\nERROR | "
                f"{lake_info['name']} | "
                f"{year}"
            )

            print(str(e))


            results[lake_id][year] = {

                'image': None,

                'probability': None,

                'mask': None,

                'area_km2': np.nan,

                'transform': None,

                'crs': None,

                'height': None,

                'width': None,

                'water_pixels': None,

                'lake_fraction': np.nan,

                'error': str(e)

            }


print("\n" + "=" * 80)
print("ALL UTTARAKHAND LAKE-YEAR COMBINATIONS PROCESSED")
print("=" * 80)

In [ ]:
print("CRS:", crs)
print("Transform:", transform)
print("Pixel size:", abs(transform.a), abs(transform.e))

In [ ]:
# ---------------------------------------------------------
# Inspect every prediction
# ---------------------------------------------------------

for lake_id, lake_info in LAKES.items():

    for year in YEARS:

        item = results[lake_id][year]

        if item['image'] is None:
            continue

        show_prediction(
            item['image'],
            item['probability'],
            item['mask'],
            lake_info['name'],
            year
        )

In [ ]:
# ---------------------------------------------------------
# Area change summary
# ---------------------------------------------------------

rows = []

for lake_id, lake_info in LAKES.items():

    row = {
        'Lake': lake_info['name']
    }

    for year in YEARS:
        row[str(year)] = results[lake_id][year]['area_km2']

    area_2019 = row.get('2019', np.nan)
    area_2024 = row.get('2024', np.nan)

    if (
        np.isfinite(area_2019)
        and area_2019 > 0
        and np.isfinite(area_2024)
    ):
        change_pct = (
            (area_2024 - area_2019)
            / area_2019
        ) * 100

    else:
        change_pct = np.nan

    row['Change_%'] = change_pct

    rows.append(row)

area_df = pd.DataFrame(rows)

print(area_df.to_string(index=False))

In [ ]:
# ---------------------------------------------------------
# Uttarakhand multi-temporal results summary
# ---------------------------------------------------------

rows = []

for lake_id, lake_info in LAKES.items():

    areas = {
        year: results[lake_id][year]['area_km2']
        for year in YEARS
    }

    area_2019 = areas.get(2021, np.nan)
    area_2024 = areas.get(2025, np.nan)

    # Absolute change
    if (
        np.isfinite(area_2019)
        and np.isfinite(area_2024)
    ):
        change_km2 = area_2024 - area_2019
    else:
        change_km2 = np.nan

    # Percentage change
    if (
        np.isfinite(area_2019)
        and area_2019 > 0
        and np.isfinite(area_2024)
    ):
        change_pct = (
            (area_2024 - area_2019)
            / area_2019
        ) * 100
    else:
        change_pct = np.nan


    rows.append({

        'Lake_ID': lake_id,

        'Lake_Name': lake_info['name'],

        'Latitude': lake_info['center'][0],

        'Longitude': lake_info['center'][1],

        'Area_2021_km2': area_2019,

        'Area_2022_km2': areas.get(2022, np.nan),

        'Area_2023_km2': areas.get(2023, np.nan),

        'Area_2024_km2': areas.get(2024, np.nan),

        'Area_2025_km2': area_2024,

        'Change_2021_2025_km2': change_km2,

        'Change_2021_2025_pct': change_pct

    })


area_df = pd.DataFrame(rows)


print("\nUTTARAKHAND GLACIAL LAKE AREA CHANGE")
print("=" * 100)

print(
    area_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

In [ ]:
# ---------------------------------------------------------
# Plot lake-area changes
# ---------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(10, 6)
)

for lake_id, lake_info in LAKES.items():

    years_available = []
    areas_available = []

    for year in YEARS:

        area = results[lake_id][year]['area_km2']

        if np.isfinite(area):

            years_available.append(year)
            areas_available.append(area)

    if len(years_available) > 0:

        ax.plot(
            years_available,
            areas_available,
            marker='o',
            linewidth=2,
            markersize=7,
            label=lake_info['name']
        )

ax.set_title(
    'Glacial Lake Area Change (2021–2025)'
)

ax.set_xlabel('Year')
ax.set_ylabel('Detected Lake Area (km²)')

ax.grid(
    alpha=0.3
)

ax.legend()

plt.tight_layout()

plt.savefig(
    'area_change_trends.png',
    dpi=200,
    bbox_inches='tight'
)

plt.show()


# Model v3

In [ ]:
import folium

# ---------------------------------------------------------
# Create interactive map
# ---------------------------------------------------------

m = folium.Map(
    location=[28.0, 87.5],
    zoom_start=7,
    tiles='OpenStreetMap'
)

# Google satellite basemap
folium.TileLayer(
    tiles='https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}',
    name='Google Satellite',
    attr='Google'
).add_to(m)


def get_color(change_pct):

    if not np.isfinite(change_pct):
        return '#757575'       # Gray = insufficient baseline

    if change_pct > 15:
        return '#d32f2f'       # High expansion

    elif change_pct > 5:
        return '#f57c00'       # Moderate expansion

    else:
        return '#388e3c'       # Stable / low change


# ---------------------------------------------------------
# Add lake markers
# ---------------------------------------------------------

for lake_id, lake_info in LAKES.items():

    areas = {
        year:
        results[lake_id][year]['area_km2']
        for year in YEARS
    }

    area_2019 = areas[2021]
    area_2024 = areas[2025]

    if (
        np.isfinite(area_2019)
        and area_2019 > 0
        and np.isfinite(area_2024)
    ):

        change_pct = (
            (area_2024 - area_2019)
            / area_2019
        ) * 100

        area_change = (
            area_2024 - area_2019
        )

    else:

        change_pct = np.nan
        area_change = np.nan

    color = get_color(change_pct)

    # Use 2024 area for marker size when available
    if np.isfinite(area_2024):

        radius = max(
            7,
            min(30, area_2024 * 15)
        )

    else:

        radius = 7

    year_rows = ''.join(
        f"""
        <tr>
            <td style="padding:4px">{year}</td>
            <td style="padding:4px;text-align:right">
                {areas[year]:.3f}
            </td>
        </tr>
        """
        for year in YEARS
        if np.isfinite(areas[year])
    )

    if np.isfinite(change_pct):

        change_text = f"""
        <b>2019 → 2024 change:</b>
        {change_pct:+.1f}%<br>
        <b>Area difference:</b>
        {area_change:+.3f} km²
        """

    else:

        change_text = """
        <b>2019 → 2024 change:</b>
        Not available
        """

    popup_html = f"""
    <div style="font-family:Arial; min-width:240px">

        <h4 style="margin-bottom:8px">
            {lake_info['name']}
        </h4>

        <table style="
            width:100%;
            border-collapse:collapse;
        ">

            <tr style="background:#eeeeee">
                <th style="padding:4px;text-align:left">
                    Year
                </th>
                <th style="padding:4px;text-align:right">
                    Area (km²)
                </th>
            </tr>

            {year_rows}

        </table>

        <div style="
            margin-top:10px;
            padding:7px;
            background:#f5f5f5;
            border-radius:5px;
        ">

            {change_text}

        </div>

    </div>
    """

    folium.CircleMarker(

        location=lake_info['center'],

        radius=radius,

        color=color,

        fill=True,

        fill_color=color,

        fill_opacity=0.65,

        weight=2,

        popup=folium.Popup(
            popup_html,
            max_width=300
        ),

        tooltip=(
            f"{lake_info['name']} | "
            + (
                f"{change_pct:+.1f}%"
                if np.isfinite(change_pct)
                else "Baseline unavailable"
            )
        )

    ).add_to(m)


# ---------------------------------------------------------
# Legend
# ---------------------------------------------------------

legend_html = """
<div style="
position: fixed;
bottom: 40px;
left: 40px;
z-index: 1000;
background: white;
padding: 12px 16px;
border-radius: 8px;
box-shadow: 0 2px 8px rgba(0,0,0,0.3);
font-family: Arial;
font-size: 13px;
">

<b>Lake Area Change</b><br><br>

<span style="color:#d32f2f">●</span>
High expansion (>15%)<br>

<span style="color:#f57c00">●</span>
Moderate expansion (5–15%)<br>

<span style="color:#388e3c">●</span>
Low / stable (<5%)<br>

<span style="color:#757575">●</span>
Baseline unavailable<br><br>

<small>
Marker size ∝ 2024 detected area<br>
Model: U-Net + Sentinel-2
</small>

</div>
"""

m.get_root().html.add_child(
    folium.Element(legend_html)
)

folium.LayerControl().add_to(m)

m.save(
    'glacial_lake_map.html'
)

print(
    "Interactive map saved as glacial_lake_map.html"
)